# Deploy Decision 3.0 (d3-mini) on SageMaker AI

Deploy [d3-mini](https://huggingface.co/vllm-sr/d3-mini), the 4B multimodal
foundation decision model of Decision 3.0, on one NVIDIA L4 real-time endpoint
(`ml.g6.2xlarge`, falling back to `ml.g6.4xlarge` and `ml.g6e.2xlarge`). Invoke its native Choice, Yes/No (`noul`), and Score outputs over text,
images, and videos through Transformers `AutoModel` and `system_one`. The
checkpoint and its remote runtime code are pinned to the same revision. Set
`DECISION_MODEL=nano` or `DECISION_MODEL=flash` to deploy
[d3-nano](https://huggingface.co/vllm-sr/d3-nano) (2B) or
[d3-flash](https://huggingface.co/vllm-sr/d3-flash) (9B) through the same cells.

It runs on the maintained AWS PyTorch 2.14 SageMaker DLC, through the small serving
image that this notebook builds with AWS CodeBuild. Read [README.md](README.md) for
prerequisites, licensing, and runtime details. This is an evaluation example.

Run from this directory. Run the cleanup cell when finished, including after a
failed deployment once SageMaker allows endpoint deletion.

## 1. Environment preparation

In [ ]:
%pip install --upgrade 'boto3>=1.43.1,<2' 'huggingface_hub>=1.0,<2' --quiet

In [ ]:
import base64
import json
import math
import os
import statistics
import tarfile
import time
import uuid
from pathlib import Path

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError
from huggingface_hub import hf_hub_download

session = boto3.Session()
region = session.region_name or "us-east-1"
sm = session.client("sagemaker", region_name=region)
runtime = session.client(
    "sagemaker-runtime", region_name=region,
    config=Config(read_timeout=70, retries={"max_attempts": 0}),
)
s3 = session.client("s3", region_name=region)
iam = session.client("iam")
logs = session.client("logs", region_name=region)
identity = session.client("sts").get_caller_identity()
account_id = identity["Account"]

# The three Decision 3.0 models share the d3 runtime, the request shape, and this adapter.
MODELS = {
    "nano": {
        "model_id": "vllm-sr/d3-nano",
        "revision": "7bdafa1db05271702efdd4fa2bd2f66a03680763",
        "model_name": "d3-nano",
        "loaded_parameters": 2213763904,
        "base_model": "Qwen/Qwen3.5-2B",
    },
    "mini": {
        "model_id": "vllm-sr/d3-mini",
        "revision": "f15a639c070dccd1da611d83c0618c77be8c23d6",
        "model_name": "d3-mini",
        "loaded_parameters": 4539918336,
        "base_model": "Qwen/Qwen3.5-4B",
    },
    "flash": {
        "model_id": "vllm-sr/d3-flash",
        "revision": "c7b9fa351ffe8d17f1f97ecacf1e36a9256bad8c",
        "model_name": "d3-flash",
        "loaded_parameters": 8393739504,
        "base_model": "Qwen/Qwen3.5-9B",
    },
}
model_choice = os.environ.get("DECISION_MODEL", "mini").lower()
if model_choice not in MODELS:
    raise ValueError(f"DECISION_MODEL must be one of {sorted(MODELS)}.")
selected = MODELS[model_choice]

# Set this when the notebook's own role is not a SageMaker execution role.
execution_role = os.environ.get("SAGEMAKER_EXECUTION_ROLE_ARN")
bucket = os.environ.get("SAGEMAKER_ARTIFACT_BUCKET", f"sagemaker-{region}-{account_id}")
# One NVIDIA L4 (24 GB) holds d3-nano and d3-mini with images and videos (d3-mini peaked
# below 10 GiB); d3-flash needs the L40S (48 GB) of ml.g6e. With more than one type, the
# endpoint uses a capacity-aware instance pool and falls back to the next type in order.
default_instances = "ml.g6e.2xlarge" if model_choice == "flash" else "ml.g6.2xlarge,ml.g6.4xlarge,ml.g6e.2xlarge"
instance_types = [
    t.strip() for t in os.environ.get("SAGEMAKER_INSTANCE_TYPES", default_instances).split(",") if t.strip()
]
if not instance_types:
    raise ValueError("SAGEMAKER_INSTANCE_TYPES must name at least one instance type.")
# Set this to reuse a serving image you already built; otherwise section 2 builds one.
inference_image = os.environ.get("SAGEMAKER_INFERENCE_IMAGE_URI")
codebuild_role_arn = os.environ.get("CODEBUILD_ROLE_ARN")
# The serving image runs PyTorch 2.14 with CUDA 13, which needs the NVIDIA driver 580 AMI.
inference_ami_version = os.environ.get("SAGEMAKER_INFERENCE_AMI_VERSION", "al2023-ami-sagemaker-inference-gpu-4-1")
# Questions per forward pass (runtime default 8) and kernel warm-up at startup (default on).
d3_batch_size = os.environ.get("D3_BATCH_SIZE", "")
d3_warmup = os.environ.get("D3_WARMUP", "")
name = f"d3-{model_choice}-{uuid.uuid4().hex[:12]}"
artifact_key = f"d3-{model_choice}/{name}/model.tar.gz"
artifact_version = None
example_dir = Path.cwd()
archive_path = example_dir / f"{name}.tar.gz"
for filename in ["inference.py", "requirements.txt", "example-request.json"]:
    if not (example_dir / filename).is_file():
        raise FileNotFoundError(f"Run the notebook from its example directory: missing {filename}")
print(f"Region: {region}; model: {selected['model_id']}; instances: {', '.join(instance_types)}; endpoint: {name}")

### Resolve the execution role

In Studio the current role may already be appropriate. Outside Studio,
set `SAGEMAKER_EXECUTION_ROLE_ARN` to an existing SageMaker role. This cell
reads the role's actual ARN through IAM, preserving paths such as
`service-role/`, and checks its SageMaker trust.

In [ ]:
if execution_role is None:
    caller_arn = identity["Arn"]
    if ":assumed-role/" not in caller_arn:
        raise ValueError("Set SAGEMAKER_EXECUTION_ROLE_ARN to an existing SageMaker execution role.")
    role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
else:
    role_name = execution_role.rsplit("/", 1)[-1]
role_info = iam.get_role(RoleName=role_name)["Role"]
if execution_role is not None and execution_role != role_info["Arn"]:
    raise ValueError("The configured execution-role ARN does not match the resolved IAM role.")
execution_role = role_info["Arn"]
trusts_sagemaker = any(
    statement.get("Effect") == "Allow"
    and "sagemaker.amazonaws.com" in (
        [statement.get("Principal", {}).get("Service")]
        if isinstance(statement.get("Principal", {}).get("Service"), str)
        else statement.get("Principal", {}).get("Service", [])
    )
    for statement in role_info["AssumeRolePolicyDocument"]["Statement"]
)
if not trusts_sagemaker:
    raise ValueError(
        "The current role does not trust SageMaker. Set SAGEMAKER_EXECUTION_ROLE_ARN "
        "to an appropriate existing execution role and rerun setup."
    )
print("Execution role resolved and SageMaker trust verified.")

## 2. Build the serving image

There is no AWS PyTorch inference DLC after 2.6, and that image is past its end of
support. This example runs on the maintained PyTorch 2.14 SageMaker DLC (CUDA 13)
through the small shared image in
[03-features/decision-model-serving-image](../../../03-features/decision-model-serving-image/).
It adds `serve`, which answers `/ping` and `/invocations` by calling this example's
`model_fn` and `transform_fn`. The [Strands Decider](../../StrandsAgents/strands-decider-2B/)
example uses the same image; reuse it with `SAGEMAKER_INFERENCE_IMAGE_URI`.

AWS CodeBuild builds the image and pushes it to a private Amazon ECR repository in your
account, in about 7 minutes. Set `CODEBUILD_ROLE_ARN` to a CodeBuild service role that can
read the artifact bucket, write build logs, and push to Amazon ECR.

In [ ]:
import sys

if inference_image is None:
    if not codebuild_role_arn:
        raise ValueError("Set CODEBUILD_ROLE_ARN, or SAGEMAKER_INFERENCE_IMAGE_URI to reuse an image.")
    image_dir = example_dir.parents[2] / "03-features" / "decision-model-serving-image"
    if not image_dir.is_dir():
        raise FileNotFoundError(f"{image_dir} not found. Run this notebook from a clone of the repository.")
    sys.path.insert(0, str(image_dir))
    from build_image import build_image

    inference_image = build_image(region=region, bucket=bucket, codebuild_role_arn=codebuild_role_arn)
print(f"Serving image: {inference_image}")

## 3. Prepare the model artifact

Package the companion handler and requirements under `code/`, as expected by
the serving image. The requirements file adds the optional `flash-linear-attention`
kernels for the Qwen3.5 Gated DeltaNet layers and OpenCV for video decoding;
everything else comes from the image.
The worker downloads the pinned checkpoint (4.4 GB for d3-nano, 9.1 GB for d3-mini,
16.8 GB for d3-flash), verifies it against `MODEL_MANIFEST.json`, and loads it with
`trust_remote_code=True`. Review the published code before adopting it for another
workload.

The default artifact bucket is shared and is preserved during cleanup. Capture
the uploaded object's version directly from the upload response.

In [ ]:
try:
    s3.head_bucket(Bucket=bucket)
except ClientError as error:
    if error.response["Error"]["Code"] not in {"404", "NoSuchBucket", "NotFound"}:
        raise
    create_args = {"Bucket": bucket}
    if region != "us-east-1":
        create_args["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**create_args)
    s3.put_public_access_block(
        Bucket=bucket,
        PublicAccessBlockConfiguration={
            "BlockPublicAcls": True, "IgnorePublicAcls": True,
            "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
        },
    )

with tarfile.open(archive_path, "w:gz") as archive:
    for filename in ["inference.py", "requirements.txt"]:
        archive.add(example_dir / filename, arcname=f"code/{filename}")
with archive_path.open("rb") as artifact:
    uploaded = s3.put_object(Bucket=bucket, Key=artifact_key, Body=artifact)
artifact_version = uploaded.get("VersionId")
model_data_url = f"s3://{bucket}/{artifact_key}"
print("Inference artifact uploaded.")

## 4. Create the model, endpoint configuration, and endpoint

SageMaker allows 1,800 seconds for the container to pass its health check, which
covers installing the kernels, downloading the checkpoint, loading it to the GPU,
and the runtime's warm-up (kernel compilation for every batch size, plus three
image requests). The serving image runs one process, so the weights load once, and
the adapter serializes inference. The runtime fixes numerics (BF16 backbone, FP32
readout); do not override its dtype.

`DECISION_MODEL_ID` and `DECISION_MODEL_REVISION` select the pinned checkpoint.
`InferenceAmiVersion` selects the NVIDIA driver 580 AMI that the CUDA 13 serving
image needs. No `VolumeSizeInGB` is set.

In [ ]:
environment = {
    "SAGEMAKER_PROGRAM": "inference.py",
    "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
    "DECISION_MODEL_ID": selected["model_id"],
    "DECISION_MODEL_REVISION": selected["revision"],
    "HF_HOME": "/tmp/huggingface",
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "TOKENIZERS_PARALLELISM": "false",
}
if d3_batch_size:
    environment["D3_BATCH_SIZE"] = d3_batch_size
if d3_warmup:
    environment["D3_WARMUP"] = d3_warmup
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=execution_role,
    PrimaryContainer={
        "Image": inference_image,
        "ModelDataUrl": model_data_url,
        "Environment": environment,
    },
)
instance_choice = (
    {"InstanceType": instance_types[0]} if len(instance_types) == 1 else {
        "InstancePools": [{"InstanceType": t, "Priority": p} for p, t in enumerate(instance_types, 1)],
        "VariantInstanceProvisionTimeoutInSeconds": 1800,
    }
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic", "ModelName": name,
        "InitialInstanceCount": 1, "InitialVariantWeight": 1.0,
        "ModelDataDownloadTimeoutInSeconds": 900,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
        "InferenceAmiVersion": inference_ami_version,
        **instance_choice,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print(f"Endpoint {name} is deploying.")

In [ ]:
# 4,800 s outlasts the configured worst case with an instance pool: 1,800 s provisioning
# + 900 s model-data download + 1,800 s container health check.
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 160}
)
endpoint = sm.describe_endpoint(EndpointName=name)
assert endpoint["EndpointStatus"] == "InService"
assert endpoint["EndpointConfigName"] == name
assert sum(v["CurrentInstanceCount"] for v in endpoint["ProductionVariants"]) == 1
variant = endpoint["ProductionVariants"][0]
# With an instance pool, the pool entry that holds the instance names the type that won.
instance_type = next(
    (pool["InstanceType"] for pool in variant.get("InstancePools", []) if pool.get("CurrentInstanceCount")),
    instance_types[0],
)
print(f"Endpoint is InService with one {instance_type} instance.")

## 5. Verify the runtime and invoke typed questions

The adapter accepts `{"state": ..., "questions": {...}, "images": [...], "videos": [...]}`
through SageMaker's IAM-authenticated API, with images and videos as base64 data URLs,
and runs the runtime's `prepare`, `run`, and `respond` steps as the package's own
`d3_server.py` does. Its diagnostics operation reports effective versions, the GPU,
which Gated DeltaNet kernels Transformers bound, the image and video contracts,
load and warm-up times, and memory use.

In [ ]:
def invoke(payload):
    response = runtime.invoke_endpoint(
        EndpointName=name, ContentType="application/json",
        Accept="application/json", Body=json.dumps(payload),
    )
    return json.loads(response["Body"].read())

def data_url(path, media_type):
    return f"data:{media_type};base64," + base64.b64encode(Path(path).read_bytes()).decode("ascii")

# The GPU each instance family carries; keeps the check meaningful when a pool chose the type.
EXPECTED_GPU = {"ml.g6e": "NVIDIA L40S", "ml.g6": "NVIDIA L4", "ml.g5": "NVIDIA A10G"}
instance_family = instance_type.rsplit(".", 1)[0]

diagnostics = invoke({"operation": "diagnostics"})
assert diagnostics["device"] == "cuda:0"
if instance_family in EXPECTED_GPU:
    assert diagnostics["gpu"] == EXPECTED_GPU[instance_family], (instance_type, diagnostics["gpu"])
else:
    print(f"Unlisted instance family {instance_family}; GPU reported as {diagnostics['gpu']!r}.")
assert diagnostics["model"] == selected["model_name"]
assert diagnostics["model_id"] == selected["model_id"]
assert diagnostics["model_revision"] == selected["revision"]
assert diagnostics["loaded_parameters"] == selected["loaded_parameters"]
assert diagnostics["loaded_parameters"] == diagnostics["manifest_parameters"]["loaded"]
assert diagnostics["source_identity"]["base_model"]["repo_id"] == selected["base_model"]
assert set(diagnostics["parameter_dtypes"]) == {"torch.bfloat16", "torch.float32"}
assert diagnostics["attention_mode"] == "noncausal_full_attention"
assert diagnostics["max_input_tokens"] is None
assert diagnostics["images"]["supported"] and diagnostics["videos"]["supported"]
assert diagnostics["versions"]["torch"].startswith("2.14.")
assert diagnostics["versions"]["transformers"] == "5.17.0"
# The pinned remote code was validated against these; a bump in the shared image shows up here.
assert diagnostics["versions"]["huggingface_hub"].startswith("1.33.")
assert diagnostics["versions"]["flash-linear-attention"] == "0.5.2"
print(json.dumps(diagnostics, indent=2))
print("Gated DeltaNet kernels:", json.dumps(diagnostics["kernels"]))

### Text: support-ticket routing with Choice, Yes/No, and Score

The same request the [Decision 2.0 Kai 0.6B CPU](../Decision-2.0-Kai-0.6B-cpu/)
example sends, so the two families can be compared on identical input. The
harness owns thresholds and what happens next.

In [ ]:
request = json.loads((example_dir / "example-request.json").read_text())
result = invoke(request)
print(json.dumps(result, indent=2))

In [ ]:
def check_answers(response, questions):
    assert response["model"] == diagnostics["model"]
    answers = response["answers"]
    assert set(answers) == set(questions)
    for key, question in questions.items():
        answer = answers[key]
        assert answer["type"] == question["type"], answer
        assert "error" not in answer, answer
        if question["type"] == "noul":
            assert math.isfinite(answer["noul"]) and 0 <= answer["noul"] <= 1
            continue
        probabilities = list(answer["probabilities"].values())
        assert all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities)
        assert abs(sum(probabilities) - 1) <= 0.001
        if question["type"] == "choice":
            assert answer["choice"] in question["criteria"]
        else:
            assert 0 <= answer["score"] <= len(question["criteria"]) - 1
    return answers

answers = check_answers(result, request["questions"])
print("All three answer types verified.")

### Image: a damaged blender and its receipt

The repository's example receipt goes to the endpoint as a base64 PNG data URL in
`images`; every question of the request sees it. The runtime reads each image at
up to 1.6 megapixels, 32 × 32 pixels per token. SageMaker's `InvokeEndpoint`
payload limit (6 MB) bounds what can be sent inline.

In [ ]:
receipt = hf_hub_download(selected["model_id"], "assets/example-receipt.png", revision=selected["revision"])
image_request = {
    "state": "The customer says the blender arrived cracked and attached the receipt.",
    "images": [data_url(receipt, "image/png")],
    "questions": {
        "route": request["questions"]["route"],
        "on_receipt": {"type": "noul", "instructions": "Does the receipt list the blender?"},
        "payment": {
            "type": "choice", "instructions": "How was the order paid?",
            "criteria": {"card": None, "cash": None, "gift card": None},
        },
    },
}
image_result = invoke(image_request)
image_answers = check_answers(image_result, image_request["questions"])
print(json.dumps({k: v for k, v in image_result.items() if k != "answers"}, indent=2))
print(json.dumps(image_answers, indent=2))

### Video: a moving square

The repository's test clip goes as a base64 MP4 data URL in `videos`. The runtime
decodes it with OpenCV at 2 frames per second (4 to 32 frames spread over the
clip, up to 0.2 megapixels each), placed after any images and before the text.

In [ ]:
clip = hf_hub_download(selected["model_id"], "assets/example-video.mp4", revision=selected["revision"])
video_request = {
    "state": "A short clip from a test camera.",
    "videos": [data_url(clip, "video/mp4")],
    "questions": {
        "direction": {
            "type": "choice", "instructions": "Which way does the square move?",
            "criteria": {"right": "From left to right", "left": "From right to left", "still": "It does not move"},
        },
        "color_change": {"type": "noul", "instructions": "Does the square change color?"},
    },
}
video_result = invoke(video_request)
video_answers = check_answers(video_result, video_request["questions"])
print(json.dumps({k: v for k, v in video_result.items() if k != "answers"}, indent=2))
print(json.dumps(video_answers, indent=2))

### Basic expected-answer smoke checks

These simple cases check the serving path. They are not a representative
benchmark of decision quality or calibration.

In [ ]:
assert answers["route"]["choice"] == "returns"
assert image_answers["route"]["choice"] == "returns"
assert image_answers["on_receipt"]["noul"] >= 0.5
assert image_answers["payment"]["choice"] == "card"
assert video_answers["direction"]["choice"] == "right"
assert video_answers["color_change"]["noul"] >= 0.5

language = invoke({
    "state": "This is a clear sentence written in English.",
    "questions": {"language": {
        "type": "choice", "instructions": "Which language is this sentence written in?",
        "criteria": {"English": "English", "French": "French", "German": "German"},
    }},
})
assert language["answers"]["language"]["choice"] == "English"

city_question = {"city_supplied": {
    "type": "noul", "instructions": "Did the user explicitly provide a city?",
}}
explicit_city = invoke({
    "state": "The user says: What is the current weather in Paris?",
    "questions": city_question,
})
missing_city = invoke({
    "state": "The user says: What is the current weather?",
    "questions": city_question,
})
assert explicit_city["answers"]["city_supplied"]["noul"] >= 0.5
assert missing_city["answers"]["city_supplied"]["noul"] < 0.5
print("Basic expected-answer checks: 9/9.")

### Repeatability and server latency

Three repeats of each modality give a hosting smoke measurement, not a production
latency estimate. `latency_ms` measures server-side inference, including media
decoding; `gpu_peak_allocated_bytes` is the allocator peak of that request. SDK
and network time is measured separately in the scaling check.

In [ ]:
for label, payload, reference in [
    ("text", request, result), ("image", image_request, image_result), ("video", video_request, video_result),
]:
    repeats = [invoke(payload) for _ in range(3)]
    assert all(item["answers"] == reference["answers"] for item in repeats)
    print(f"{label}: median warm server latency {statistics.median(item['latency_ms'] for item in repeats):.2f} ms, "
          f"GPU peak {max(item['gpu_peak_allocated_bytes'] for item in repeats) / 2**30:.2f} GiB")

### Malformed requests are refused before anything runs

The adapter follows the package server: an unknown question type, an unknown
field, or media that is not a valid base64 data URL is HTTP 422 with a `detail`
body. Images and videos are accepted only as data URLs, never as file paths or
web addresses, so the endpoint does not read its own disk or fetch from the
network on a caller's behalf. SageMaker wraps the error in `ModelError` and may
omit the body.

In [ ]:
def expect_model_error(payload, status):
    try:
        invoke(payload)
    except runtime.exceptions.ModelError as error:
        assert error.response.get("OriginalStatusCode") == status, error.response
        return
    raise AssertionError(f"Expected model HTTP {status}.")

expect_model_error({"state": "hello", "questions": {"q": {"type": "bogus", "instructions": "x"}}}, 422)
expect_model_error({"state": "hello", "questions": city_question, "extra": 1}, 422)
expect_model_error({"state": "hello", "questions": city_question, "images": ["https://example.com/a.png"]}, 422)
expect_model_error({"state": "hello", "questions": city_question, "images": ["data:image/png;base64,AAAA"]}, 422)
print("Invalid question, unknown field, web-address image, and undecodable image rejected with HTTP 422.")

### Long state is not truncated

The checkpoint declares no input limit (`max_input_tokens` is null) on a backbone
with 262,144 positions, so a long state is answered in full rather than rejected
or truncated. Latency and GPU memory grow with the prompt; this request is about
15,000 tokens. Keep requests inside SageMaker's 60-second invocation limit and
bound state length in the harness.

In [ ]:
started = time.perf_counter()
long_result = invoke({"state": "weather city location " * 5000, "questions": city_question})
round_trip_ms = (time.perf_counter() - started) * 1000
assert long_result["usage"]["input_tokens"] > 10000
assert "error" not in long_result["answers"]["city_supplied"]
print(f"{long_result['usage']['input_tokens']} input tokens: server {long_result['latency_ms']:.0f} ms, "
      f"round trip {round_trip_ms:.0f} ms, GPU peak {long_result['gpu_peak_allocated_bytes'] / 2**30:.2f} GiB")

## 6. Bounded multi-question scaling

Run 1, 8, 32, and 64 independent Yes/No question IDs over the same short state.
These are repeatability and hosting checks, not a decision-quality benchmark.
The runtime decides one question per forward pass and runs `batch_size`
questions (default 8) per pass, left-padded to the longest prompt.

Measure three requests per question count. Server latency excludes SDK/network
time; round-trip latency includes it. GPU peak allocation covers the request;
worker RSS is the lifetime high-water mark. Neither metric includes every
process on the hosting instance. These results apply only to this short input,
runtime, batch size, and instance.

In [ ]:
scaling_results = []
for count in [1, 8, 32, 64]:
    payload = {
        "state": "The user explicitly requested the current weather in Paris.",
        "questions": {
            f"city_{index}": {
                "type": "noul",
                "instructions": "Did the user explicitly provide a city?",
            }
            for index in range(count)
        },
    }
    server_times, round_trip_times, peaks = [], [], []
    baseline_answers = None
    for _ in range(3):
        started = time.perf_counter()
        response = invoke(payload)
        round_trip_times.append((time.perf_counter() - started) * 1000)
        assert set(response["answers"]) == set(payload["questions"])
        for answer in response["answers"].values():
            assert answer["type"] == "noul" and math.isfinite(answer["noul"])
            assert 0 <= answer["noul"] <= 1
        if baseline_answers is None:
            baseline_answers = response["answers"]
        else:
            assert response["answers"] == baseline_answers
        server_times.append(response["latency_ms"])
        peaks.append(response["gpu_peak_allocated_bytes"])
    memory = invoke({"operation": "diagnostics"})
    scaling_results.append({
        "questions": count,
        "input_tokens": response["usage"]["input_tokens"],
        "server_median_ms": round(statistics.median(server_times), 2),
        "round_trip_median_ms": round(statistics.median(round_trip_times), 2),
        "gpu_peak_allocated_bytes": max(peaks),
        "worker_peak_rss_bytes": memory["worker_peak_rss_bytes"],
        "worker_peak_rss_scope": "worker_lifetime",
    })
print(json.dumps({
    "model": memory["model"],
    "instance_type": instance_type,
    "gpu": memory["gpu"],
    "kernels": memory["kernels"],
    "batch_size": memory["batch_size"],
    "results": scaling_results,
}, indent=2))

## 7. Cleanup

Delete this example's endpoint and wait before deleting its configuration and
model. Remove every S3 version and delete marker for this example's unique
artifact key, including repeated uploads. Preserve the shared bucket and role.
Attempt independent cleanup even if another operation fails; report failures.

The serving image, its Amazon ECR repository (`decision-model-serving`), and the
CodeBuild project are kept so later runs can reuse them; see the README to
remove them. Late endpoint log delivery can recreate the CloudWatch log group
after deletion. Rerun cleanup after delivery settles if the group reappears.

In [ ]:
def ignore_missing(operation, **arguments):
    try:
        return operation(**arguments)
    except ClientError as error:
        message = error.response["Error"].get("Message", "")
        error_code = error.response["Error"]["Code"]
        if error_code == "ResourceNotFoundException" or (
            error_code == "ValidationException"
            and ("Could not find" in message or "does not exist" in message)
        ):
            return None
        raise

def delete_endpoint():
    if ignore_missing(sm.describe_endpoint, EndpointName=name) is not None:
        sm.delete_endpoint(EndpointName=name)
        sm.get_waiter("endpoint_deleted").wait(
            EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 120}
        )

def artifact_versions():
    return [
        {"Key": item["Key"], "VersionId": item["VersionId"]}
        for page in s3.get_paginator("list_object_versions").paginate(
            Bucket=bucket, Prefix=artifact_key
        )
        for group in ("Versions", "DeleteMarkers")
        for item in page.get(group, [])
        if item["Key"] == artifact_key
    ]

def delete_artifact():
    for item in artifact_versions():
        s3.delete_object(Bucket=bucket, **item)
    if artifact_versions():
        raise RuntimeError("S3 artifact versions remain after cleanup.")

cleanup_errors = []
def attempt_cleanup(label, operation, *positional, **arguments):
    try:
        operation(*positional, **arguments)
        return True
    except Exception as error:
        cleanup_errors.append(f"{label}: {error}")
        return False

try:
    if attempt_cleanup("endpoint", delete_endpoint):
        attempt_cleanup(
            "endpoint configuration", ignore_missing,
            sm.delete_endpoint_config, EndpointConfigName=name,
        )
        attempt_cleanup("model", ignore_missing, sm.delete_model, ModelName=name)
    attempt_cleanup("S3 artifact", delete_artifact)
    attempt_cleanup(
        "endpoint logs", ignore_missing,
        logs.delete_log_group, logGroupName=f"/aws/sagemaker/Endpoints/{name}",
    )
finally:
    attempt_cleanup("local archive", archive_path.unlink, missing_ok=True)

if cleanup_errors:
    raise RuntimeError("Cleanup incomplete: " + "; ".join(cleanup_errors))
assert ignore_missing(sm.describe_endpoint, EndpointName=name) is None
assert ignore_missing(sm.describe_endpoint_config, EndpointConfigName=name) is None
assert ignore_missing(sm.describe_model, ModelName=name) is None
print("Endpoint, configuration, model, all artifact versions, and endpoint logs cleaned up.")